# MHE (Multiple Hash Embeddings) — Criteo Benchmark
**Paper:** "Revisiting the Performance of iALS on Item Recommendation Benchmarks" / 
Hash Embeddings variants (2021)

## Architecture Overview
- **MHE**: Dùng T bảng embedding nhỏ, mỗi category được hash vào mỗi bảng → lấy T vectors → weighted sum → embedding 64 chiều
- **DeepFM & DCN**: 13 integer features → `log1p` normalize; 26 cat features → MHE → 64-dim
- **DLRM**: 13 integer features → MLP → 64-dim; 26 cat features → MHE → 64-dim
- **Loss**: BCEWithLogitsLoss (raw logits, không sigmoid trong model)
- **Metric**: AUC-ROC

## MHE vs DHE
| | DHE | MHE |
|---|---|---|
| Approach | Hash code → MLP | T hash tables → weighted sum |
| Parameters | MLP weights (shared across features) | T × B × dim tables |
| Memory | Low (no large tables) | Configurable (B much smaller than vocab) |
| Speed | Slower (MLP forward) | Faster (lookup + sum) |
| Collision handling | Learned via MLP | Reduced by using multiple tables |

In [18]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import os, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(f'PyTorch: {torch.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

PyTorch: 2.10.0+cu128
Device : cuda


In [19]:
# ============================================================
# Cell 2 — CONFIG
# ============================================================
CFG = {
    # --- Data ---
    'data_dir'    : '/kaggle/input/datasets/huy291/criteo-cleaned-data/data',
    'num_files'   : 3,         # ← Đặt số file muốn load (1-50). None = full 50 files
    'val_ratio'   : 0.1,
    'test_ratio'  : 0.1,

    # --- MHE params ---
    'embed_dim'   : 64,        # output embedding dimension
    'mhe_num_tables': 4,       # T: số bảng hash (thường 2-8)
    'mhe_bucket_size': 1000000,# B: kích thước mỗi bảng (có thể nhỏ hơn vocab)
    # Lưu ý: MHE hiệu quả nhất khi vocab >> bucket_size
    # Nếu vocab < bucket_size, MHE sẽ tự điều chỉnh xuống min(vocab, bucket_size)

    # --- Integer features (cho DLRM) ---
    'int_mlp_dims': [64, 64],

    # --- Training ---
    'batch_size'  : 4096,
    'epochs'      : 5,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,

    # --- Models ---
    'deepfm_hidden'    : [400, 400, 400],
    'dcn_cross_layers' : 3,
    'dcn_hidden'       : [512, 512],
    'dlrm_top_mlp'     : [512, 256, 128],
}

DENSE_COLS = [f'I{i}' for i in range(1, 14)]
CAT_COLS   = [f'C{i}' for i in range(1, 27)]
LABEL_COL  = 'label'
print('Config loaded.')

Config loaded.


In [20]:
# ============================================================
# Cell 3 — Load Data
# ============================================================
def load_data(data_dir, num_files=None):
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.parquet')])
    if num_files is not None:
        files = files[:num_files]
    print(f'Loading {len(files)} parquet files...')
    dfs = []
    for i, f in enumerate(files):
        df = pd.read_parquet(os.path.join(data_dir, f))
        dfs.append(df)
        if (i+1) % 5 == 0:
            print(f'  {i+1}/{len(files)} files loaded')
    df = pd.concat(dfs, ignore_index=True)
    print(f'Total shape: {df.shape}')
    return df

df = load_data(CFG['data_dir'], CFG['num_files'])
print(df.head(2))

Loading 3 parquet files...
Total shape: (2764159, 40)
   label  I1  I2  I3  I4    I5  I6  I7  I8   I9  ...       C17       C18  \
0      0   1   1   5   0  1382   4  15   2  181  ...  e5ba7672  f54016b9   
1      0   2   0  44   1   102   8   2   2    4  ...  07c540c4  b04e4670   

        C19       C20       C21      C22       C23       C24       C25  \
0  21ddcdc9  b1252a9d  07b5194c  unknown  3a171ecb  c5c50484  e8b83407   
1  21ddcdc9  5840adea  60f6221e  unknown  3a171ecb  43f13e8b  e8b83407   

        C26  
0  9727dd16  
1  731c3655  

[2 rows x 40 columns]


In [21]:
# ============================================================
# Cell 4 — Label Encode
# ============================================================
vocab_sizes = []
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    vocab_sizes.append(int(df[col].max()) + 1)

df[DENSE_COLS] = df[DENSE_COLS].astype(np.float32)
df[LABEL_COL]  = df[LABEL_COL].astype(np.float32)

# MHE bucket size: min(vocab, mhe_bucket_size) per feature
mhe_buckets = [min(v, CFG['mhe_bucket_size']) for v in vocab_sizes]
print('Vocab sizes (first 5):', vocab_sizes[:5])
print('MHE buckets (first 5):', mhe_buckets[:5])

Vocab sizes (first 5): [1388, 542, 800816, 257436, 288]
MHE buckets (first 5): [1388, 542, 800816, 257436, 288]


In [22]:
# ============================================================
# Cell 5 — Train/Val/Test Split
# ============================================================
n = len(df)
test_size  = int(n * CFG['test_ratio'])
val_size   = int(n * CFG['val_ratio'])
train_size = n - test_size - val_size

train_df = df.iloc[:train_size].reset_index(drop=True)
val_df   = df.iloc[train_size:train_size+val_size].reset_index(drop=True)
test_df  = df.iloc[train_size+val_size:].reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

Train: 2,211,329  Val: 276,415  Test: 276,415


In [23]:
# ============================================================
# Cell 6 — Dataset & DataLoader
# ============================================================
class CriteoDataset(Dataset):
    def __init__(self, df, dense_cols, cat_cols, label_col):
        self.dense = torch.tensor(df[dense_cols].values, dtype=torch.float32)
        self.cat   = torch.tensor(df[cat_cols].values,   dtype=torch.long)
        self.label = torch.tensor(df[label_col].values,  dtype=torch.float32)

    def __len__(self): return len(self.label)
    def __getitem__(self, idx): return self.dense[idx], self.cat[idx], self.label[idx]

train_ds = CriteoDataset(train_df, DENSE_COLS, CAT_COLS, LABEL_COL)
val_ds   = CriteoDataset(val_df,   DENSE_COLS, CAT_COLS, LABEL_COL)
test_ds  = CriteoDataset(test_df,  DENSE_COLS, CAT_COLS, LABEL_COL)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],   shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=4, pin_memory=True)
print('DataLoaders ready.')

DataLoaders ready.


In [24]:
# ============================================================
# Cell 7 — MHE Module
# ============================================================
# Multiple Hash Embeddings:
# Mỗi categorical feature dùng T embedding tables nhỏ (bucket size B << vocab size).
# Với mỗi table t, hash id vào bucket → lookup → ra vector dim/T.
# T vectors được concat → linear projection → embed_dim.
#
# Ưu điểm:
# - Giảm memory: T * B * d thay vì vocab * d
# - Mỗi collision pattern khác nhau giữa các bảng → ít bị nhiễu hơn single hash
# - Differentiable, trainable hoàn toàn

class MHEEncoder(nn.Module):
    """
    Multiple Hash Embedding cho một categorical feature.
    Input : (B,) LongTensor
    Output: (B, embed_dim) FloatTensor
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 num_tables: int = 4, bucket_size: int = 1_000_000):
        super().__init__()
        self.num_tables  = num_tables
        self.bucket_size = min(bucket_size, vocab_size)  # không cần bảng lớn hơn vocab
        self.embed_dim   = embed_dim

        # sub_dim: mỗi table output sub_dim chiều
        # Sau đó concat T vectors → T*sub_dim → project về embed_dim
        # Chọn sub_dim = embed_dim để concat rồi project (giữ thông tin phong phú)
        self.sub_dim = embed_dim

        # T embedding tables, mỗi bảng kích thước bucket_size × sub_dim
        self.tables = nn.ModuleList([
            nn.Embedding(self.bucket_size, self.sub_dim)
            for _ in range(num_tables)
        ])

        # Learnable weights cho weighted sum của T tables
        self.table_weights = nn.Parameter(torch.ones(num_tables) / num_tables)

        # Projection từ sub_dim → embed_dim (nếu muốn transform)
        # Với weighted sum, sub_dim = embed_dim nên projection là identity
        # Thêm LayerNorm để ổn định
        self.norm = nn.LayerNorm(embed_dim)

        # Các hash parameters (prime-based universal hashing)
        # Để deterministic và stable, dùng fixed random primes
        torch.manual_seed(1337)
        self.register_buffer('a_vec', torch.randint(1, 2**31 - 1, (num_tables,), dtype=torch.long))
        self.register_buffer('b_vec', torch.randint(0, 2**31 - 1, (num_tables,), dtype=torch.long))
        self.prime = 2**31 - 1

    def _hash(self, ids: torch.Tensor, t: int) -> torch.Tensor:
        """Hash ids vào bucket [0, bucket_size) dùng universal hashing."""
        return ((self.a_vec[t] * ids + self.b_vec[t]) % self.prime) % self.bucket_size

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        # ids: (B,)
        weights = F.softmax(self.table_weights, dim=0)  # (T,)
        emb = torch.zeros(ids.size(0), self.embed_dim, device=ids.device)
        for t in range(self.num_tables):
            hashed = self._hash(ids, t)             # (B,)
            e = self.tables[t](hashed)               # (B, sub_dim)
            emb = emb + weights[t] * e
        return self.norm(emb)  # (B, embed_dim)


class MHELayer(nn.Module):
    """
    Wrapper áp dụng MHE cho tất cả 26 cat features.
    Input : (B, 26) LongTensor
    Output: (B, 26, embed_dim)
    """
    def __init__(self, vocab_sizes, embed_dim, num_tables, bucket_size):
        super().__init__()
        self.encoders = nn.ModuleList([
            MHEEncoder(v, embed_dim, num_tables, bucket_size)
            for v in vocab_sizes
        ])
        self.embed_dim = embed_dim

        total_params = sum(sum(p.numel() for p in enc.parameters()) for enc in self.encoders)
        print(f'  MHELayer: {len(vocab_sizes)} features, '
              f'total embedding params: {total_params:,}')

    def forward(self, cat_x: torch.Tensor) -> torch.Tensor:
        embs = [enc(cat_x[:, i]) for i, enc in enumerate(self.encoders)]
        return torch.stack(embs, dim=1)  # (B, 26, D)

print('MHE module defined.')

MHE module defined.


In [25]:
# ============================================================
# Cell 8 — Numeric Feature Processors (same as DHE notebook)
# ============================================================
class Log1pNormalizer(nn.Module):
    """log(1+|x|)*sign(x) + LayerNorm — cho DeepFM & DCN integer features."""
    def __init__(self, num_features):
        super().__init__()
        self.norm = nn.LayerNorm(num_features)

    def forward(self, x):
        return self.norm(torch.log1p(torch.abs(x)) * torch.sign(x))


class DenseEmbedMLP(nn.Module):
    """
    Embed 13 integer features thành (B, 13, embed_dim) cho DLRM.
    Shared MLP across features.
    """
    def __init__(self, num_dense, embed_dim, hidden_dims):
        super().__init__()
        layers = []
        in_d = 1
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)
        self.log_norm = Log1pNormalizer(1)
        self.num_dense = num_dense

    def forward(self, x):
        out = []
        for i in range(self.num_dense):
            xi = self.log_norm(x[:, i:i+1])
            out.append(self.mlp(xi))
        return torch.stack(out, dim=1)  # (B, 13, embed_dim)

print('Numeric processors defined.')

Numeric processors defined.


In [26]:
# ============================================================
# Cell 9 — DeepFM with MHE  (FIXED: logits, BCEWithLogitsLoss)
# ============================================================
class DeepFM_MHE(nn.Module):
    """
    DeepFM với MHE embedding.
    FIX: output raw logits, dùng BCEWithLogitsLoss.
    FM terms được normalize để tránh gradient explosion.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, hidden_dims,
                 mhe_num_tables, mhe_bucket_sizes):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)

        self.dense_norm = Log1pNormalizer(num_dense)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables,
                            CFG['mhe_bucket_size'])  # MHE tự điều chỉnh per feature

        # FM order-1
        self.fm1_sparse_proj = nn.Linear(embed_dim, 1, bias=False)
        self.fm1_dense_proj  = nn.Linear(num_dense, 1)

        # FM order-2 norm
        self.fm2_norm = nn.LayerNorm(1)

        # Deep
        deep_in = num_dense + self.num_sparse * embed_dim
        layers = []
        in_d = deep_in
        for dim in hidden_dims:
            layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        layers.append(nn.Linear(in_d, 1))
        self.deep = nn.Sequential(*layers)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)  # (B, 13)
        emb = self.mhe(sparse_x)               # (B, 26, 64)

        # FM-1
        fm1 = self.fm1_sparse_proj(emb).squeeze(-1).sum(1, keepdim=True) + \
              self.fm1_dense_proj(dense_norm)  # (B,1)

        # FM-2 (sparse only)
        sum_emb = emb.sum(1)
        fm2 = 0.5 * (sum_emb**2 - (emb**2).sum(1)).sum(1, keepdim=True)  # (B,1)
        fm2 = self.fm2_norm(fm2)

        # Deep
        deep_in = torch.cat([dense_norm, emb.view(dense_x.size(0), -1)], dim=1)
        deep_out = self.deep(deep_in)

        return fm1 + fm2 + deep_out  # raw logit

print('DeepFM_MHE defined.')

DeepFM_MHE defined.


In [27]:
# ============================================================
# Cell 10 — DCN with MHE  (FIXED)
# ============================================================
class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.w = nn.ParameterList([nn.Parameter(torch.empty(input_dim)) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])
        for w in self.w:
            nn.init.xavier_uniform_(w.unsqueeze(0))

    def forward(self, x0):
        xl = x0
        for i in range(self.num_layers):
            xlw = (xl * self.w[i]).sum(1, keepdim=True)
            xl  = x0 * xlw + self.b[i] + xl
        return xl


class DCN_MHE(nn.Module):
    """
    DCN với MHE embedding.
    FIX: bỏ sigmoid trong model.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, cross_layers, hidden_dims,
                 mhe_num_tables, mhe_bucket_size):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_norm = Log1pNormalizer(num_dense)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables, mhe_bucket_size)

        input_dim = num_dense + self.num_sparse * embed_dim  # 13 + 26*64 = 1677
        self.cross_net = CrossNetwork(input_dim, cross_layers)

        deep_layers = []
        in_d = input_dim
        for dim in hidden_dims:
            deep_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        self.deep_net = nn.Sequential(*deep_layers)
        self.fc = nn.Linear(input_dim + in_d, 1)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)
        emb = self.mhe(sparse_x)                          # (B, 26, 64)
        emb_flat = emb.view(dense_x.size(0), -1)          # (B, 1664)
        x0 = torch.cat([dense_norm, emb_flat], dim=1)     # (B, 1677)

        cross_out = self.cross_net(x0)
        deep_out  = self.deep_net(x0)
        combined  = torch.cat([cross_out, deep_out], dim=1)
        return self.fc(combined)  # raw logit

print('DCN_MHE defined.')

DCN_MHE defined.


In [28]:
# ============================================================
# Cell 11 — DLRM with MHE
# ============================================================
class DLRM_MHE(nn.Module):
    """
    DLRM với MHE cho sparse features, MLP embed cho dense features.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim,
                 dense_embed_hidden, top_mlp_dims,
                 mhe_num_tables, mhe_bucket_size):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_embed = DenseEmbedMLP(num_dense, embed_dim, dense_embed_hidden)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables, mhe_bucket_size)

        num_all = num_dense + self.num_sparse          # 39
        num_interactions = (num_all * (num_all - 1)) // 2
        top_in = embed_dim + num_interactions

        top_layers = []
        in_d = top_in
        for dim in top_mlp_dims:
            top_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        top_layers.append(nn.Linear(in_d, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        dense_emb  = self.dense_embed(dense_x)   # (B, 13, 64)
        sparse_emb = self.mhe(sparse_x)           # (B, 26, 64)
        all_emb    = torch.cat([dense_emb, sparse_emb], dim=1)  # (B, 39, 64)

        Z = torch.bmm(all_emb, all_emb.transpose(1, 2))  # (B, 39, 39)
        n = all_emb.size(1)
        rows, cols = torch.triu_indices(n, n, offset=1)
        interactions = Z[:, rows, cols]  # (B, num_interactions)

        dense_rep = dense_emb.mean(dim=1)  # (B, 64)
        concat = torch.cat([dense_rep, interactions], dim=1)
        return self.top_mlp(concat)  # raw logit

print('DLRM_MHE defined.')

DLRM_MHE defined.


In [29]:
# ============================================================
# Cell 12 — Training Engine
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logit = model(dense, cat).squeeze(1)
                loss  = criterion(logit, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logit = model(dense, cat).squeeze(1)
            loss  = criterion(logit, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * label.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        logit = model(dense, cat).squeeze(1)
        loss  = criterion(logit, label)
        total_loss += loss.item() * label.size(0)
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader.dataset), roc_auc_score(labels, preds)


def run_experiment(model_name, model, cfg, train_dl, val_dl, test_dl, device):
    print(f'\n{"="*60}')
    print(f'  Model: {model_name}  |  Embedding: MHE')
    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {nparams:,}')
    print(f'{"="*60}')

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg['lr'], steps_per_epoch=len(train_dl), epochs=cfg['epochs']
    )
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_auc = 0
    results = []
    for epoch in range(1, cfg['epochs'] + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_dl, optimizer, criterion, device, scaler)
        scheduler.step()
        val_loss, val_auc = evaluate(model, val_dl, criterion, device)
        elapsed = time.time() - t0
        print(f'  Epoch {epoch}/{cfg["epochs"]} | Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | {elapsed:.1f}s')
        results.append({'epoch': epoch, 'train_loss': train_loss,
                         'val_loss': val_loss, 'val_auc': val_auc})
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), f'{model_name}_MHE_best.pt')

    model.load_state_dict(torch.load(f'{model_name}_MHE_best.pt'))
    test_loss, test_auc = evaluate(model, test_dl, criterion, device)
    print(f'\n   Test AUC: {test_auc:.4f}  |  Test Loss: {test_loss:.4f}')
    return results, test_auc

print('Training engine ready.')

Training engine ready.


In [30]:
# ============================================================
# Cell 13 — Memory Analysis: MHE vs Standard Embedding
# ============================================================
print('MHE vs Standard Embedding — Memory Analysis')
print('-' * 60)
embed_dim   = CFG['embed_dim']
num_tables  = CFG['mhe_num_tables']
bucket_size = CFG['mhe_bucket_size']

total_std, total_mhe = 0, 0
for i, (col, vs) in enumerate(zip(CAT_COLS, vocab_sizes)):
    std_params = vs * embed_dim
    b = min(bucket_size, vs)
    mhe_params = num_tables * b * embed_dim
    ratio = mhe_params / std_params if std_params > 0 else 1
    total_std += std_params
    total_mhe += mhe_params
    if i < 5 or vs > 100000:
        print(f'  {col}: vocab={vs:,}, std={std_params:,}, mhe={mhe_params:,}, ratio={ratio:.2f}x')

print(f'\n  TOTAL  std={total_std:,}, mhe={total_mhe:,}, ratio={total_mhe/total_std:.2f}x')
print(f'  Savings: {(1 - total_mhe/total_std)*100:.1f}%' if total_mhe < total_std else '  MHE uses more memory for small vocabs — bucket_size auto-adjusted.')

MHE vs Standard Embedding — Memory Analysis
------------------------------------------------------------
  C1: vocab=1,388, std=88,832, mhe=355,328, ratio=4.00x
  C2: vocab=542, std=34,688, mhe=138,752, ratio=4.00x
  C3: vocab=800,816, std=51,252,224, mhe=205,008,896, ratio=4.00x
  C4: vocab=257,436, std=16,475,904, mhe=65,903,616, ratio=4.00x
  C5: vocab=288, std=18,432, mhe=73,728, ratio=4.00x
  C12: vocab=671,339, std=42,965,696, mhe=171,862,784, ratio=4.00x
  C16: vocab=495,249, std=31,695,936, mhe=126,783,744, ratio=4.00x
  C21: vocab=595,181, std=38,091,584, mhe=152,366,336, ratio=4.00x

  TOTAL  std=193,635,264, mhe=774,541,056, ratio=4.00x
  MHE uses more memory for small vocabs — bucket_size auto-adjusted.


In [31]:
# ============================================================
# Cell 14 — Run DeepFM + MHE
# ============================================================
deepfm_model = DeepFM_MHE(
    num_dense      = len(DENSE_COLS),
    vocab_sizes    = vocab_sizes,
    embed_dim      = CFG['embed_dim'],
    hidden_dims    = CFG['deepfm_hidden'],
    mhe_num_tables = CFG['mhe_num_tables'],
    mhe_bucket_sizes = vocab_sizes,  # MHELayer xử lý per-feature
)
deepfm_results, deepfm_test_auc = run_experiment(
    'DeepFM', deepfm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

  MHELayer: 26 features, total embedding params: 774,544,488

  Model: DeepFM  |  Embedding: MHE
  Params: 775,539,395
  Epoch 1/5 | Train Loss: 0.6037 | Val Loss: 0.4926 | Val AUC: 0.7370 | 44.5s
  Epoch 2/5 | Train Loss: 0.4947 | Val Loss: 0.4826 | Val AUC: 0.7507 | 44.4s
  Epoch 3/5 | Train Loss: 0.4832 | Val Loss: 0.4787 | Val AUC: 0.7580 | 44.4s
  Epoch 4/5 | Train Loss: 0.4761 | Val Loss: 0.4745 | Val AUC: 0.7616 | 44.7s
  Epoch 5/5 | Train Loss: 0.4710 | Val Loss: 0.4726 | Val AUC: 0.7643 | 44.1s

   Test AUC: 0.7656  |  Test Loss: 0.4755


In [32]:
# ============================================================
# Cell 15 — Run DCN + MHE
# ============================================================
dcn_model = DCN_MHE(
    num_dense      = len(DENSE_COLS),
    vocab_sizes    = vocab_sizes,
    embed_dim      = CFG['embed_dim'],
    cross_layers   = CFG['dcn_cross_layers'],
    hidden_dims    = CFG['dcn_hidden'],
    mhe_num_tables = CFG['mhe_num_tables'],
    mhe_bucket_size= CFG['mhe_bucket_size'],
)
dcn_results, dcn_test_auc = run_experiment(
    'DCN', dcn_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

  MHELayer: 26 features, total embedding params: 774,544,488

  Model: DCN  |  Embedding: MHE
  Params: 775,680,606
  Epoch 1/5 | Train Loss: 0.5322 | Val Loss: 0.4862 | Val AUC: 0.7474 | 44.2s
  Epoch 2/5 | Train Loss: 0.4801 | Val Loss: 0.4767 | Val AUC: 0.7595 | 44.4s
  Epoch 3/5 | Train Loss: 0.4712 | Val Loss: 0.4721 | Val AUC: 0.7651 | 44.0s
  Epoch 4/5 | Train Loss: 0.4655 | Val Loss: 0.4703 | Val AUC: 0.7686 | 43.9s
  Epoch 5/5 | Train Loss: 0.4608 | Val Loss: 0.4688 | Val AUC: 0.7708 | 44.1s

   Test AUC: 0.7720  |  Test Loss: 0.4719


In [33]:
# ============================================================
# Cell 16 — Run DLRM + MHE
# ============================================================
dlrm_model = DLRM_MHE(
    num_dense          = len(DENSE_COLS),
    vocab_sizes        = vocab_sizes,
    embed_dim          = CFG['embed_dim'],
    dense_embed_hidden = CFG['int_mlp_dims'],
    top_mlp_dims       = CFG['dlrm_top_mlp'],
    mhe_num_tables     = CFG['mhe_num_tables'],
    mhe_bucket_size    = CFG['mhe_bucket_size'],
)
dlrm_results, dlrm_test_auc = run_experiment(
    'DLRM', dlrm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

  MHELayer: 26 features, total embedding params: 774,544,488

  Model: DLRM  |  Embedding: MHE
  Params: 775,132,011
  Epoch 1/5 | Train Loss: 0.5390 | Val Loss: 0.5190 | Val AUC: 0.6840 | 46.7s
  Epoch 2/5 | Train Loss: 0.5191 | Val Loss: 0.5120 | Val AUC: 0.6987 | 46.6s
  Epoch 3/5 | Train Loss: 0.5127 | Val Loss: 0.5091 | Val AUC: 0.7056 | 46.6s
  Epoch 4/5 | Train Loss: 0.5080 | Val Loss: 0.5063 | Val AUC: 0.7094 | 46.3s
  Epoch 5/5 | Train Loss: 0.5046 | Val Loss: 0.5041 | Val AUC: 0.7131 | 46.5s

   Test AUC: 0.7142  |  Test Loss: 0.5076


In [34]:
# ============================================================
# Cell 17 — Summary & Comparison
# ============================================================
print('\n' + '='*60)
print('  FINAL RESULTS — MHE Embedding')
print('='*60)
print(f'  DeepFM-MHE  Test AUC: {deepfm_test_auc:.4f}')
print(f'  DCN-MHE     Test AUC: {dcn_test_auc:.4f}')
print(f'  DLRM-MHE    Test AUC: {dlrm_test_auc:.4f}')
print('='*60)

summary = pd.DataFrame({
    'model'     : ['DeepFM-MHE', 'DCN-MHE', 'DLRM-MHE'],
    'embedding' : ['MHE'] * 3,
    'test_auc'  : [deepfm_test_auc, dcn_test_auc, dlrm_test_auc],
    'num_files' : [CFG['num_files']] * 3,
})





  FINAL RESULTS — MHE Embedding
  DeepFM-MHE  Test AUC: 0.7656
  DCN-MHE     Test AUC: 0.7720
  DLRM-MHE    Test AUC: 0.7142
